1、生成0-255对应映射

2、将语料按特殊tokens划分 train_segments

In [1]:
import regex as re
import os
from collections import defaultdict, Counter
import regex as re  # type: ignore
import json

special_tokens = ["<|endoftext|>"]
text = "Hello World World<|endoftext|>Hello Heppy happy<|endoftext|>!"

special_regex = "|".join(re.escape(t) for t in special_tokens)
parts = re.split(f"({special_regex})", text)
train_segments = [p for p in parts if p not in special_tokens]

print(parts)
print(train_segments)

['Hello World World', '<|endoftext|>', 'Hello Heppy happy', '<|endoftext|>', '!']
['Hello World World', 'Hello Heppy happy', '!']


3、预分词：GPT2正则表达式

In [2]:
from collections import Counter

gpt2_pat = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

raw_counts = Counter()

for segment in train_segments:
        # 对每个语料片段应用预分词正则，找到所有“单词”
    words = gpt2_pat.findall(segment)
    for word in words:
        raw_counts[tuple(bytes([b]) for b in word.encode("utf-8"))] += 1

print(raw_counts)


Counter({(b'H', b'e', b'l', b'l', b'o'): 2, (b' ', b'W', b'o', b'r', b'l', b'd'): 2, (b' ', b'H', b'e', b'p', b'p', b'y'): 1, (b' ', b'h', b'a', b'p', b'p', b'y'): 1, (b'!',): 1})


In [3]:
words_list = []
counts_list = []
c = []
for word_tuple, freq in raw_counts.items():
    words_list.append(list(word_tuple)) # 转换为 list 以便后面修改
    counts_list.append(freq)
print(words_list)
print(counts_list)

[[b'H', b'e', b'l', b'l', b'o'], [b' ', b'W', b'o', b'r', b'l', b'd'], [b' ', b'H', b'e', b'p', b'p', b'y'], [b' ', b'h', b'a', b'p', b'p', b'y'], [b'!']]
[2, 2, 1, 1, 1]


In [4]:
stats = defaultdict(int)
indices = defaultdict(set)

for idx, word in enumerate(words_list):
    freq = counts_list[idx] # 获取该单词的出现频率
    for i in range(len(word) - 1):
        pair = (word[i], word[i+1])
        stats[pair] += freq          # 累加该 pair 的全局频率
        indices[pair].add(idx)       # 将当前单词的索引加入该 pair 的倒排列表中
print(stats)
print(indices)

defaultdict(<class 'int'>, {(b'H', b'e'): 3, (b'e', b'l'): 2, (b'l', b'l'): 2, (b'l', b'o'): 2, (b' ', b'W'): 2, (b'W', b'o'): 2, (b'o', b'r'): 2, (b'r', b'l'): 2, (b'l', b'd'): 2, (b' ', b'H'): 1, (b'e', b'p'): 1, (b'p', b'p'): 2, (b'p', b'y'): 2, (b' ', b'h'): 1, (b'h', b'a'): 1, (b'a', b'p'): 1})
defaultdict(<class 'set'>, {(b'H', b'e'): {0, 2}, (b'e', b'l'): {0}, (b'l', b'l'): {0}, (b'l', b'o'): {0}, (b' ', b'W'): {1}, (b'W', b'o'): {1}, (b'o', b'r'): {1}, (b'r', b'l'): {1}, (b'l', b'd'): {1}, (b' ', b'H'): {2}, (b'e', b'p'): {2}, (b'p', b'p'): {2, 3}, (b'p', b'y'): {2, 3}, (b' ', b'h'): {3}, (b'h', b'a'): {3}, (b'a', b'p'): {3}})


4、合并

In [6]:
merges = [] 
num_merges=3

for _ in range(num_merges):
    if not stats:
        break      
    best_pair = max(stats.items(), key=lambda x: (x[1], x[0]))[0]    
    if stats[best_pair] <= 0:
        break        
    
    merges.append(best_pair)
    new_token = best_pair[0] + best_pair[1]
    relevant_indices = list(indices[best_pair])
    
    

In [ ]:
print(relevant_indices)

[0, 2]
